# Recommendation System

In [ ]:
#importiing libraries
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler

In [ ]:
df = pd.read_csv('/content/anime.csv')

df.head()

,anime_id,name,genre,type,episodes,rating,members
0,32281,Kimi no Na wa.,"Drama, Romance, School, Supernatural",Movie,1,9.37,200630
1,5114,Fullmetal Alchemist: Brotherhood,"Action, Adventure, Drama, Fantasy, Magic, Mili...",TV,64,9.26,793665
2,28977,Gintama°,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.25,114262
3,9253,Steins;Gate,"Sci-Fi, Thriller",TV,24,9.17,673572
4,9969,Gintama&#039;,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.16,151266


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12294 entries, 0 to 12293
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   anime_id  12294 non-null  int64  
 1   name      12294 non-null  object 
 2   genre     12232 non-null  object 
 3   type      12269 non-null  object 
 4   episodes  12294 non-null  object 
 5   rating    12064 non-null  float64
 6   members   12294 non-null  int64  
dtypes: float64(1), int64(2), object(4)
memory usage: 672.5+ KB


In [ ]:
df.shape

(12294, 7)

In [ ]:
df.describe()

,anime_id,rating,members
count,12294.000000,12064.000000,1.229400e+04
mean,14058.221653,6.473902,1.807134e+04
std,11455.294701,1.026746,5.482068e+04
min,1.000000,1.670000,5.000000e+00
25%,3484.250000,5.880000,2.250000e+02
50%,10260.500000,6.570000,1.550000e+03
75%,24794.500000,7.180000,9.437000e+03
max,34527.000000,10.000000,1.013917e+06


In [ ]:
df.isnull().sum()

,0
anime_id,0
name,0
genre,62
type,25
episodes,0
rating,230
members,0


In [ ]:
df.duplicated().sum()

np.int64(0)

In [ ]:
#handling missing values
df['genre'] = df['genre'].fillna('Unknown')

df['type'] = df['type'].fillna(df['type'].mode()[0])

df['rating'] = df['rating'].fillna(df['rating'].median())

df.isnull().sum()

,0
anime_id,0
name,0
genre,0
type,0
episodes,0
rating,0
members,0


In [ ]:
#Convert Episodes Column
df['episodes'] = df['episodes'].replace('Unknown', np.nan)

df['episodes'] = pd.to_numeric(df['episodes'])

df['episodes'] = df['episodes'].fillna(df['episodes'].median())

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

#Normalize Numerical Features
scaler = MinMaxScaler()

# Re-clean 'episodes' column to ensure it is fully numeric
df['episodes'] = df['episodes'].replace('Unknown', np.nan)
df['episodes'] = pd.to_numeric(df['episodes'], errors='coerce') # Added errors='coerce' for robustness
df['episodes'] = df['episodes'].fillna(df['episodes'].median())

df[['rating','members','episodes']] = scaler.fit_transform(
    df[['rating','members','episodes']]
)

df.head()

,anime_id,name,genre,type,episodes,rating,members
0,32281,Kimi no Na wa.,"Drama, Romance, School, Supernatural",Movie,0.000000,0.924370,0.197872
1,5114,Fullmetal Alchemist: Brotherhood,"Action, Adventure, Drama, Fantasy, Magic, Mili...",TV,0.034673,0.911164,0.782770
2,28977,Gintama°,"Action, Comedy, Historical, Parody, Samurai, S...",TV,0.027518,0.909964,0.112689
3,9253,Steins;Gate,"Sci-Fi, Thriller",TV,0.012658,0.900360,0.664325
4,9969,Gintama&#039;,"Action, Comedy, Historical, Parody, Samurai, S...",TV,0.027518,0.899160,0.149186


In [ ]:
#Feature Extraction
tfidf = TfidfVectorizer(stop_words='english')

genre_matrix = tfidf.fit_transform(df['genre'])

genre_matrix.shape

(12294, 47)

In [ ]:
#Cosine Similarity
cosine_sim = cosine_similarity(genre_matrix)

cosine_sim.shape

(12294, 12294)

In [ ]:
#Create Index Mapping
indices = pd.Series(df.index,index=df['name']).drop_duplicates()

indices.head()

,0
name,
Kimi no Na wa.,0
Fullmetal Alchemist: Brotherhood,1
Gintama°,2
Steins;Gate,3
Gintama&#039;,4


In [ ]:
#Create Index Mapping
indices = pd.Series(df.index,index=df['name']).drop_duplicates()

indices.head()

,0
name,
Kimi no Na wa.,0
Fullmetal Alchemist: Brotherhood,1
Gintama°,2
Steins;Gate,3
Gintama&#039;,4


In [ ]:
#Recommendation Function
def recommend_anime(title, cosine_sim=cosine_sim):

    idx = indices[title]

    sim_scores = list(enumerate(cosine_sim[idx]))

    sim_scores = sorted(sim_scores,key=lambda x:x[1],reverse=True)

    sim_scores = sim_scores[1:11]

    anime_indices = [i[0] for i in sim_scores]

    return df[['name','genre','rating']].iloc[anime_indices]

In [ ]:
#Test Recommendation
recommend_anime('Death Note')

,name,genre,rating
778,Death Note Rewrite,"Mystery, Police, Psychological, Supernatural, ...",0.740696
981,Mousou Dairinin,"Drama, Mystery, Police, Psychological, Superna...",0.728691
144,Higurashi no Naku Koro ni Kai,"Mystery, Psychological, Supernatural, Thriller",0.809124
1383,Higurashi no Naku Koro ni Rei,"Comedy, Mystery, Psychological, Supernatural, ...",0.707083
445,Mirai Nikki (TV),"Action, Mystery, Psychological, Shounen, Super...",0.768307
4164,Mirai Nikki (TV): Ura Mirai Nikki,"Action, Comedy, Mystery, Psychological, Shoune...",0.614646
334,Higurashi no Naku Koro ni,"Horror, Mystery, Psychological, Supernatural, ...",0.780312
38,Monster,"Drama, Horror, Mystery, Police, Psychological,...",0.846339
5382,AD Police,"Adventure, Dementia, Mecha, Mystery, Police, P...",0.576230
2074,Higurashi no Naku Koro ni Kaku: Outbreak,"Horror, Mystery, Psychological, Thriller",0.683073


In [ ]:
recommend_anime('Naruto')

,name,genre,rating
615,Naruto: Shippuuden,"Action, Comedy, Martial Arts, Shounen, Super P...",0.752701
841,Naruto,"Action, Comedy, Martial Arts, Shounen, Super P...",0.737095
1103,Boruto: Naruto the Movie - Naruto ga Hokage ni...,"Action, Comedy, Martial Arts, Shounen, Super P...",0.721489
1343,Naruto x UT,"Action, Comedy, Martial Arts, Shounen, Super P...",0.709484
1472,Naruto: Shippuuden Movie 4 - The Lost Tower,"Action, Comedy, Martial Arts, Shounen, Super P...",0.703481
1573,Naruto: Shippuuden Movie 3 - Hi no Ishi wo Tsu...,"Action, Comedy, Martial Arts, Shounen, Super P...",0.699880
2458,Naruto Shippuuden: Sunny Side Battle,"Action, Comedy, Martial Arts, Shounen, Super P...",0.671068
2997,Naruto Soyokazeden Movie: Naruto to Mashin to ...,"Action, Comedy, Martial Arts, Shounen, Super P...",0.653061
7628,Kyutai Panic Adventure!,"Action, Martial Arts, Shounen, Super Power",0.424970
784,Naruto: Shippuuden Movie 6 - Road to Ninja,"Action, Adventure, Martial Arts, Shounen, Supe...",0.740696


In [ ]:
#Recommendation with Threshold
def recommend_threshold(title,threshold=0.40):

    idx = indices[title]

    sim_scores = list(enumerate(cosine_sim[idx]))

    sim_scores = [i for i in sim_scores if i[1]>=threshold]

    sim_scores = sorted(sim_scores,key=lambda x:x[1],reverse=True)

    anime_indices = [i[0] for i in sim_scores[1:]]

    return df[['name','genre','rating']].iloc[anime_indices]

In [ ]:
#Test Threshold
recommend_threshold('Death Note',0.50)

,name,genre,rating
778,Death Note Rewrite,"Mystery, Police, Psychological, Supernatural, ...",0.740696
981,Mousou Dairinin,"Drama, Mystery, Police, Psychological, Superna...",0.728691
144,Higurashi no Naku Koro ni Kai,"Mystery, Psychological, Supernatural, Thriller",0.809124
1383,Higurashi no Naku Koro ni Rei,"Comedy, Mystery, Psychological, Supernatural, ...",0.707083
445,Mirai Nikki (TV),"Action, Mystery, Psychological, Shounen, Super...",0.768307
...,...,...,...
1781,C: The Money of Soul and Possibility Control,"Action, Mystery, Super Power, Thriller",0.692677
493,Higashi no Eden,"Action, Comedy, Drama, Mystery, Romance, Sci-F...",0.763505
975,Higashi no Eden Movie I: The King of Eden,"Comedy, Drama, Mystery, Romance, Slice of Life...",0.728691
540,Serial Experiments Lain,"Dementia, Drama, Mystery, Psychological, Sci-F...",0.758703


In [ ]:
#Performance Analysis
print("Number of Anime :",len(df))

print("Vocabulary Size :",len(tfidf.get_feature_names_out()))

print("Cosine Similarity Matrix :",cosine_sim.shape)

Number of Anime : 12294
Vocabulary Size : 47
Cosine Similarity Matrix : (12294, 12294)


## Cluster Analysis / Comments

Insights

Cosine Similarity recommends anime based on genre similarity.
Anime having similar genres receive higher similarity scores.
Increasing the threshold gives fewer but more relevant recommendations.
Lower threshold values produce more recommendations but may reduce relevance.

**Conclusion**

The recommendation system successfully recommends similar anime using cosine similarity on genre features. TF-IDF converts text genres into numerical vectors, and cosine similarity measures how similar two anime are. The system can be improved further by incorporating user ratings, popularity, and collaborative filtering techniques.

**Interview Questions**
1. *Difference between User-Based and Item-Based Collaborative Filtering*
User-Based	Item-Based
Recommends items liked by similar users	Recommends items similar to those already liked
Compares users	Compares items
Slower for large datasets	Faster and more scalable
Example: Users with similar tastes like the same anime	Example: If you like Naruto, recommend One Piece


2. *What is Collaborative Filtering?*

Collaborative filtering is a recommendation technique that suggests items based on user behavior and preferences. It assumes that users with similar interests will like similar items. There are two types:

User-Based Collaborative Filtering – Finds users with similar preferences.
Item-Based Collaborative Filtering – Finds items that are similar based on user ratings.

This code covers all the tasks requested: preprocessing, feature extraction, cosine similarity, recommendation function, threshold experiments, performance analysis, and interview questions.